# Trabajo Práctico Final: Solución de IA Integral
**Materia:** Taller de Lenguajes de Programación III - Python para Ciencia de Datos

**Integrantes:** Escalante Facundo, Bogado Augusto, Rossi Fabiana.

## PASO 1: ENTENDIMIENTO, LIMPIEZA Y EDA

In [1]:
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

In [ ]:
df = pd.read_csv("data/student-lifestyle-and-stress-dataset.csv")
df.isnull().describe()

FileNotFoundError: [Errno 2] No such file or directory: 'student-lifestyle-and-stress-dataset.csv'

In [ ]:
df.isnull().info()

**Variable objetivo (y):** stress_level

**Características (X):** Columnas restantes

In [ ]:
df.isnull().sum()

Se identifica que las columnas nulas en el dataset son de aproximadamente 6% de los datos totales. Debido a la baja proporción de los mismos, se pueden eliminar sin problema pues no se considera que afectará al dataset final.

In [ ]:
df_copia = df.copy()
df_copia = df_copia.dropna()

In [ ]:
df_copia.isnull().sum()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(x='Stress_Level', y='Sleep_Hours', hue='Student_Type', data=df, palette='Set2')

plt.title('Distribución de Horas de Sueño por Nivel de Estrés y Tipo de Estudiante')
plt.xlabel('Nivel de Estrés')
plt.ylabel('Horas de Sueño')
plt.tight_layout()
plt.show()

## PASO 2: INGENIERÍA DE CARACTERÍSTICAS Y TRANSFORMACIÓN

Aplicamos la propiedad .get_dummies() para las variables nominales in orden jerárquico, para que nuestro dataset sea conformado solamente por datos numéricos.

In [ ]:
df_copia = pd.get_dummies(df, columns=['Student_Type'], prefix='student_type', dtype=int)
df_copia

Separamios en X y en y lo que queremos predecir y lo dividimos en entrenamiento y testeo.

In [ ]:
X = df_copia.drop(columns=['Stress_Level'])
y = df_copia['Stress_Level']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Utilizamos StandardScaler para normalizar las escalas de tiempo para evitar los sesgos por magnitud.

In [ ]:
columnas_a_escalar = ['Study_Hours', 'Sleep_Hours']

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train[columnas_a_escalar] = scaler.fit_transform(X_train[columnas_a_escalar])

In [ ]:
X_test[columnas_a_escalar] = scaler.transform(X_test[columnas_a_escalar])

## PASO 3: MODELADO

Modelamos con el RandomForestClassifier

In [ ]:
model = RandomForestClassifier(
    n_estimators= 150,
    max_depth=10,
    min_samples_split=5,
    class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

## PASO 4: EVALUACIÓN E INTERPRETACIÓN

Empezamos con la evaluación del modelo previamente hecho.

In [ ]:
y_pred = model.predict(X_test)

Hacemos un reporte de clasificación del modelo.

In [ ]:
print("--- REPORTE DE CLASIFICACIÓN ---")
print(classification_report(y_test, y_pred))

Dibujamos un gráfico del modelo final.

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión')
plt.ylabel('Realidad')
plt.xlabel('Predicción de la IA')
plt.show()

## PASO 5: EXPORTACIÓN FINAL

Exportamos el modelo final para utilizarlo en la pàgina web.

In [ ]:
joblib.dump(model, 'models/modelo_final.pkl')
joblib.dump(scaler, 'models/scaler.pkl')